In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [2]:
DB_USER='root'
DB_PASSWORD='OJlehXTRlEa6^zsFNILjnVoew9ku=E'
DB_NAME='essidb'
DB_PORT="5432"
DB_HOST='read-only-powerbi.cktgjnvajmd8.us-east-1.rds.amazonaws.com'
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

In [3]:
registros_app = pd.read_sql_query(f"""SELECT 
--u.username as "Usuario",
e.numero_doc_ident as "DNI",
--e.primer_nombre as "PrimerNombre",
--e.segundo_nombre as "SegundoNombre",
--e.apellido_paterno as "ApellidoPaterno",
--e.apellido_materno as "ApellidoMaterno",
e.fecha_nacimiento as"FechaNacimiento",
--u.date_create as "FechaCreacion",
--p.cod_centro as "CodigoCentro",
--c.descripcion as "CentroAsistencial",
--c.cod_red as "CodigoRed",
--r.descripcion as "Red",
--o.descripcion as "Operador",
c2.nro_celular  as "Celular"
--c2.email as "Correo"
from usuario u
left join paciente p on u.id_usuario = p.id_paciente
left join centro c on c.cen_asi_cod = p.cod_centro
left join red r on r.cod_red = c.cod_red
left join persona e on e.id_persona = p.id_paciente
left join contacto c2 on c2.id_persona= e.id_persona
--left join operador_movil o on c2.id_operador_movil = o.id_operador_movil
WHERE e.numero_doc_ident is not null""", con=connection1)

In [4]:
registros_app

,DNI,FechaNacimiento,Celular
0,76372969,02/08/1996,935798593
1,41218150,21/04/1982,975228227
2,41744845,23/12/1982,976595157
3,70428181,11/04/2001,957491815
4,41317315,23/04/1982,963737374
...,...,...,...
588262,43614658,22/06/1986,958667589
588263,09507246,08/05/1970,959163995
588264,72316780,19/08/1996,962716531
588265,43632739,14/06/1986,930538277


In [4]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

query = f"""
SELECT 
    p.perdocidennum,
    p.perubigeodom
FROM CMPER10 p
"""
base1 = pd.read_sql_query(query, con=connection0)


connection0.close()

In [5]:
base1=base1.rename(columns={"perdocidennum":"DNI"})
base1=base1.rename(columns={"perubigeodom":"Ubigeo"})
registros_app1=pd.merge(registros_app,base1,how='inner',on='DNI')

In [6]:
archivo_txt = '/home/ugadingenieria01/Documentos/GCTIC/DW_GCTIC/MI_CONSULTA/TB_UBIGEOS.csv'
ubigeos = pd.read_csv(archivo_txt, delimiter=';', header=0, encoding='utf-8')

In [5]:
# Obtener la fecha actual
fecha_actual = datetime.now()

registros_app['FechaNacimiento'] = pd.to_datetime(registros_app['FechaNacimiento'])

# Calcular la edad de cada individuo
registros_app['Edad'] = (fecha_actual - registros_app['FechaNacimiento']).astype('<m8[Y]')

# Definir los límites y etiquetas de los grupos etarios
#limites_grupos_etarios = [0, 11, 17, 29, 59, float('inf')]
#etiquetas_grupos_etarios = ['Niño (0-11)', 'Adolescente (12-17)', 'Joven (18-29)', 'Adulto (30-59)', 'Adulto Mayor (>=60)']

# Asignar grupos etarios a cada individuo
#registros_app['Grupo_Etario'] = pd.cut(registros_app['Edad'], bins=limites_grupos_etarios, labels=etiquetas_grupos_etarios, right=False)

/tmp/ipykernel_2462859/2497592168.py:4: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  registros_app['FechaNacimiento'] = pd.to_datetime(registros_app['FechaNacimiento'])


In [8]:
ubigeos=ubigeos.rename(columns={"ubigeo_inei":"Ubigeo"})

In [9]:
ubigeos['Ubigeo'] = ubigeos['Ubigeo'].astype('str')

In [10]:
registros_app2=pd.merge(registros_app1,ubigeos,how='left',on='Ubigeo')

In [11]:
# Realizar el merge
registros_app2 = pd.merge(registros_app1, ubigeos, how='left', on='Ubigeo', indicator=True)

# Reemplazar valores nulos en la columna 'Ubigeo' con 'No registra'
registros_app2['Ubigeo'] = registros_app2['Ubigeo'].fillna('No registra')

# Reemplazar valores no coincidentes en el merge con 'No registra'
registros_app2.loc[registros_app2['_merge'] == 'left_only', 'Ubigeo'] = 'No registra'

# Eliminar la columna '_merge' si no es necesaria
registros_app2.drop(columns=['_merge'], inplace=True)

In [6]:
# Reemplazar valores nulos en la columna 'Ubigeo' con 'No registra'
registros_app['Celular'] = registros_app['Celular'].fillna('No registra')

In [69]:
registros_app2.to_excel('info_usuarios.xlsx', index=False)

In [8]:
registros_app

,DNI,FechaNacimiento,Celular,Edad
0,76372969,1996-02-08,935798593,28.0
1,41218150,1982-04-21,975228227,42.0
2,41744845,1982-12-23,976595157,41.0
3,70428181,2001-11-04,957491815,22.0
4,41317315,1982-04-23,963737374,42.0
...,...,...,...,...
588262,43614658,1986-06-22,958667589,37.0
588263,09507246,1970-08-05,959163995,53.0
588264,72316780,1996-08-19,962716531,27.0
588265,43632739,1986-06-14,930538277,37.0


In [71]:
# Filtrar los registros que tienen un valor distinto de 'No registra' en la columna 'Celular'
registros_con_celular = registros_app2[registros_app2['Celular'] != 'No registra']

# Reemplazar valores vacíos en las columnas 'Ubigeo' y 'Departamento' con 'No registra'
registros_con_celular['Ubigeo'].fillna('No registra', inplace=True)
registros_con_celular['departamento'].fillna('No registra', inplace=True)

# Contar la cantidad de registros por departamento y grupo etario
cantidad_por_departamento_grupo_etario = registros_con_celular.groupby(['departamento', 'Grupo_Etario']).size()

# Convertir la Serie a DataFrame
df_cantidad_por_departamento_grupo_etario = cantidad_por_departamento_grupo_etario.reset_index()
df_cantidad_por_departamento_grupo_etario.columns = ['Departamento', 'Grupo_Etario', 'Cantidad']

# Ordenar el DataFrame por los nombres de los departamentos y los grupos etarios en orden descendente
df_cantidad_por_departamento_grupo_etario = df_cantidad_por_departamento_grupo_etario.sort_values(by=['Departamento', 'Grupo_Etario'], ascending=True)

df_cantidad_por_departamento_grupo_etario


,Departamento,Grupo_Etario,Cantidad
0,HUANUCO,Niño (0-11),0
1,HUANUCO,Adolescente (12-17),0
2,HUANUCO,Joven (18-29),798
3,HUANUCO,Adulto (30-59),2708
4,HUANUCO,Adulto Mayor (>=60),345
...,...,...,...
80,UCAYALI,Niño (0-11),0
81,UCAYALI,Adolescente (12-17),0
82,UCAYALI,Joven (18-29),954
83,UCAYALI,Adulto (30-59),2306


In [72]:
# Filtrar los registros que tienen un valor distinto de 'No registra' en la columna 'Celular'
registros_con_celular = registros_app2[registros_app2['Celular'] != 'No registra']
registros_con_celular_lima = registros_con_celular[registros_con_celular['provincia'] == 'LIMA']

# Reemplazar valores vacíos en las columnas 'Ubigeo' y 'Departamento' con 'No registra'
registros_con_celular_lima['Ubigeo'].fillna('No registra', inplace=True)
registros_con_celular_lima['distrito'].fillna('No registra', inplace=True)

# Contar la cantidad de registros por distrito y grupo etario
cantidad_por_distrito_grupo_etario = registros_con_celular_lima.groupby(['distrito', 'Grupo_Etario']).size()

# Convertir la Serie a DataFrame
df_cantidad_por_distrito_grupo_etario = cantidad_por_distrito_grupo_etario.reset_index()
df_cantidad_por_distrito_grupo_etario.columns = ['Distrito', 'Grupo_Etario', 'Cantidad']

# Ordenar el DataFrame por la cantidad de registros
df_cantidad_por_distrito_grupo_etario = df_cantidad_por_distrito_grupo_etario.sort_values(by=['Distrito', 'Cantidad'], ascending=False)

df_cantidad_por_distrito_grupo_etario


/tmp/ipykernel_3626500/272785643.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  registros_con_celular_lima['Ubigeo'].fillna('No registra', inplace=True)
/tmp/ipykernel_3626500/272785643.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  registros_con_celular_lima['distrito'].fillna('No registra', inplace=True)


,Distrito,Grupo_Etario,Cantidad
213,VILLA MARIA DEL TRIUNFO,Adulto (30-59),2281
212,VILLA MARIA DEL TRIUNFO,Joven (18-29),1811
214,VILLA MARIA DEL TRIUNFO,Adulto Mayor (>=60),135
210,VILLA MARIA DEL TRIUNFO,Niño (0-11),0
211,VILLA MARIA DEL TRIUNFO,Adolescente (12-17),0
...,...,...,...
3,ANCON,Adulto (30-59),611
2,ANCON,Joven (18-29),201
4,ANCON,Adulto Mayor (>=60),134
0,ANCON,Niño (0-11),0


In [ ]:
registros_app2

In [29]:
connection1.close()
engine1.dispose()

In [73]:
df_cantidad_por_departamento_grupo_etario.to_excel('info_usuarios_departamentos.xlsx', index=False)
df_cantidad_por_distrito_grupo_etario.to_excel('info_usuarios_lima.xlsx', index=False)

In [75]:
registros_app = pd.read_excel('info_usuarios.xlsx')

In [20]:
registros_app

,DNI,FechaNacimiento,Celular,Edad,Grupo_Etario
0,72035057,2001-06-27,960921462,22.0,Joven (18-29)
1,70012495,1991-09-06,906951767,32.0,Adulto (30-59)
2,08725406,1937-04-24,971825061,87.0,Adulto Mayor (>=60)
3,77165996,2002-08-25,922428399,21.0,Joven (18-29)
4,48106955,1993-03-25,933499338,31.0,Adulto (30-59)
...,...,...,...,...,...
585375,76774917,1996-03-18,926212800,28.0,Joven (18-29)
585376,16672801,1970-02-04,978233072,54.0,Adulto (30-59)
585377,29484073,1949-12-08,958808602,74.0,Adulto Mayor (>=60)
585378,46284596,1990-03-26,974863403,34.0,Adulto (30-59)


In [21]:
asegurados1 = pd.read_csv('AURELIO_DIAZ_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados2 = pd.read_csv('CHOSICA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados3 = pd.read_csv('GUSTAVO_LANATTA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados4 = pd.read_csv('HERMANA_MARIA_DONROSE_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados5 = pd.read_csv('INDEPENDENCIA_202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados6 = pd.read_csv('MARIO_MOLINA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados7 = pd.read_csv('PARAMONGA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')

/tmp/ipykernel_3502383/3830339291.py:1: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  asegurados1 = pd.read_csv('AURELIO_DIAZ_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
/tmp/ipykernel_3502383/3830339291.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  asegurados2 = pd.read_csv('CHOSICA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
/tmp/ipykernel_3502383/3830339291.py:4: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  asegurados4 = pd.read_csv('HERMANA_MARIA_DONROSE_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
/tmp/ipykernel_3502383/3830339291.py:6: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  asegurados6 = pd.read_csv('MARIO_MOLINA_P202404.csv', d

In [22]:
asegurados1=asegurados1.rename(columns={"NRODOCUMEN":"DNI"})
asegurados2=asegurados2.rename(columns={"NRODOCUMEN":"DNI"})
asegurados3=asegurados3.rename(columns={"NRODOCUMEN":"DNI"})
asegurados4=asegurados4.rename(columns={"NRODOCUMEN":"DNI"})
asegurados5=asegurados5.rename(columns={"NRODOCUMEN":"DNI"})
asegurados6=asegurados6.rename(columns={"NRODOCUMEN":"DNI"})
asegurados7=asegurados7.rename(columns={"NRODOCUMEN":"DNI"})

In [23]:
asegurados1['DNI']=asegurados1['DNI'].astype('str')
asegurados2['DNI']=asegurados2['DNI'].astype('str')
asegurados3['DNI']=asegurados3['DNI'].astype('str')
asegurados4['DNI']=asegurados4['DNI'].astype('str')
asegurados5['DNI']=asegurados5['DNI'].astype('str')
asegurados6['DNI']=asegurados6['DNI'].astype('str')
asegurados7['DNI']=asegurados7['DNI'].astype('str')

In [24]:
asegurados1 = pd.merge(asegurados1,registros_app,how='inner',on='DNI')
asegurados2 = pd.merge(asegurados2,registros_app,how='inner',on='DNI')
asegurados3 = pd.merge(asegurados3,registros_app,how='inner',on='DNI')
asegurados4 = pd.merge(asegurados4,registros_app,how='inner',on='DNI')
asegurados5 = pd.merge(asegurados5,registros_app,how='inner',on='DNI')
asegurados6 = pd.merge(asegurados6,registros_app,how='inner',on='DNI')
asegurados7 = pd.merge(asegurados7,registros_app,how='inner',on='DNI')

asegurados1 = pd.read_csv('AURELIO_DIAZ_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados2 = pd.read_csv('CHOSICA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados3 = pd.read_csv('GUSTAVO_LANATTA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados4 = pd.read_csv('HERMANA_MARIA_DONROSE_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados5 = pd.read_csv('INDEPENDENCIA_202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados6 = pd.read_csv('MARIO_MOLINA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
asegurados7 = pd.read_csv('PARAMONGA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')

In [9]:
base = pd.read_csv('CENTROS_ASIST_PB_202404_MAYORES_30.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
base=base.rename(columns={"NRODOCUMEN":"DNI"})

In [10]:
chimbote = base[base['CENTRO'] == 'CHIMBOTE']
skrabonja = base[base['CENTRO'] == 'ANTONIO SKRABONJA ANTOSICH']
torrealva = base[base['CENTRO'] == 'FELIX TORREALVA G.']
toche = base[base['CENTRO'] == 'RENE TOCHE GROPPO']
esperanza = base[base['CENTRO'] == 'LA ESPERANZA']
reategui = base[base['CENTRO'] == 'JORGE REATEGUI DELGADO']

In [11]:
chimbote ['DNI']=chimbote ['DNI'].astype('str')
skrabonja ['DNI']=skrabonja ['DNI'].astype('str')
torrealva ['DNI']=torrealva ['DNI'].astype('str')
toche ['DNI']=toche ['DNI'].astype('str')
esperanza ['DNI']=esperanza ['DNI'].astype('str')
reategui ['DNI']=reategui ['DNI'].astype('str')

/tmp/ipykernel_2462859/123302177.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chimbote ['DNI']=chimbote ['DNI'].astype('str')
/tmp/ipykernel_2462859/123302177.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  skrabonja ['DNI']=skrabonja ['DNI'].astype('str')
/tmp/ipykernel_2462859/123302177.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyd

In [12]:
chimbote = pd.merge(chimbote,registros_app,how='inner',on='DNI')
skrabonja = pd.merge(skrabonja,registros_app,how='inner',on='DNI')
torrealva = pd.merge(torrealva,registros_app,how='inner',on='DNI')
toche = pd.merge(toche,registros_app,how='inner',on='DNI')
esperanza = pd.merge(esperanza,registros_app,how='inner',on='DNI')
reategui = pd.merge(reategui,registros_app,how='inner',on='DNI')

In [77]:
# Filtrar los registros que tienen un valor distinto de 'No registra' en la columna 'Celular'
registros_con_celular = asegurados7[asegurados7['Celular'] != 'No registra']

# Reemplazar valores vacíos en las columnas 'Ubigeo' y 'Departamento' con 'No registra'
registros_con_celular['Ubigeo'].fillna('No registra', inplace=True)
registros_con_celular['departamento'].fillna('No registra', inplace=True)

# Contar la cantidad de registros por departamento y grupo etario
cantidad_por_departamento_grupo_etario = registros_con_celular.groupby(['departamento', 'Grupo_Etario']).size()

# Convertir la Serie a DataFrame
df_cantidad_por_departamento_grupo_etario = cantidad_por_departamento_grupo_etario.reset_index()
df_cantidad_por_departamento_grupo_etario.columns = ['Departamento', 'Grupo_Etario', 'Cantidad']

# Ordenar el DataFrame por los nombres de los departamentos y los grupos etarios en orden descendente
df_cantidad_por_departamento_grupo_etario = df_cantidad_por_departamento_grupo_etario.sort_values(by=['Departamento', 'Grupo_Etario'], ascending=True)

df_cantidad_por_departamento_grupo_etario


,Departamento,Grupo_Etario,Cantidad
0,LIMA,Niño (0-11),0
1,LIMA,Adolescente (12-17),0
2,LIMA,Joven (18-29),46
3,LIMA,Adulto (30-59),163
4,LIMA,Adulto Mayor (>=60),85
5,No registra,Niño (0-11),0
6,No registra,Adolescente (12-17),0
7,No registra,Joven (18-29),0
8,No registra,Adulto (30-59),4
9,No registra,Adulto Mayor (>=60),0


In [78]:
# Filtrar los registros que tienen un valor distinto de 'No registra' en la columna 'Celular'
registros_con_celular = asegurados7[asegurados7['Celular'] != 'No registra']
registros_con_celular_lima = registros_con_celular[registros_con_celular['provincia'] == 'LIMA']

# Reemplazar valores vacíos en las columnas 'Ubigeo' y 'Departamento' con 'No registra'
registros_con_celular_lima['Ubigeo'].fillna('No registra', inplace=True)
registros_con_celular_lima['distrito'].fillna('No registra', inplace=True)

# Contar la cantidad de registros por distrito y grupo etario
cantidad_por_distrito_grupo_etario = registros_con_celular_lima.groupby(['distrito', 'Grupo_Etario']).size()

# Convertir la Serie a DataFrame
df_cantidad_por_distrito_grupo_etario = cantidad_por_distrito_grupo_etario.reset_index()
df_cantidad_por_distrito_grupo_etario.columns = ['Distrito', 'Grupo_Etario', 'Cantidad']

# Ordenar el DataFrame por la cantidad de registros
df_cantidad_por_distrito_grupo_etario = df_cantidad_por_distrito_grupo_etario.sort_values(by=['Distrito', 'Cantidad'], ascending=False)

df_cantidad_por_distrito_grupo_etario


/tmp/ipykernel_1281170/1486459439.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  registros_con_celular_lima['Ubigeo'].fillna('No registra', inplace=True)
/tmp/ipykernel_1281170/1486459439.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  registros_con_celular_lima['distrito'].fillna('No registra', inplace=True)


,Distrito,Grupo_Etario,Cantidad
13,VILLA MARIA DEL TRIUNFO,Adulto (30-59),1
10,VILLA MARIA DEL TRIUNFO,Niño (0-11),0
11,VILLA MARIA DEL TRIUNFO,Adolescente (12-17),0
12,VILLA MARIA DEL TRIUNFO,Joven (18-29),0
14,VILLA MARIA DEL TRIUNFO,Adulto Mayor (>=60),0
8,SAN MIGUEL,Adulto (30-59),1
5,SAN MIGUEL,Niño (0-11),0
6,SAN MIGUEL,Adolescente (12-17),0
7,SAN MIGUEL,Joven (18-29),0
9,SAN MIGUEL,Adulto Mayor (>=60),0


In [79]:
df_cantidad_por_departamento_grupo_etario.to_excel('PARAMONGA_departamentos.xlsx', index=False)
df_cantidad_por_distrito_grupo_etario.to_excel('PARAMONGA_lima.xlsx', index=False)

In [12]:
asegurados1

,CODTIPODOC,DESDOCUMENT,DNI,AP_PATERNO_TRA,AP_MATERNO_TRA,NOMBRES_TRA,DGAFNAC,DES_SEXO,DGACECA,EDAD,DES_TIPO_ASEGURADO,NIVEL,DESCCAA,FechaNacimiento,Celular,Edad,Grupo_Etario
0,1,D.N.I.,76754827,VILCHEZ,SABARIAN,CAMILA XIOMARA,20/08/2005,MUJER,TITULAR,18,OBLIGATORIO,H_I,AURELIO DIAZ UFANO Y PERAL,2005-08-20,989461960,18.0,Joven (18-29)
1,1,D.N.I.,74831999,UBIA,PEDRAZA,FARDY JORKAEF,15/07/2005,HOMBRE,TITULAR,18,OBLIGATORIO,H_I,AURELIO DIAZ UFANO Y PERAL,2005-07-15,922780296,18.0,Joven (18-29)
2,1,D.N.I.,73087587,QUISPE,SUSAYA,LIZ VICTORIA,18/05/2005,MUJER,TITULAR,18,OBLIGATORIO,H_I,AURELIO DIAZ UFANO Y PERAL,2005-05-18,902592776,18.0,Joven (18-29)
3,1,D.N.I.,77469747,ROSALES,RUIZ,JOHANEL ARON,7/08/2005,HOMBRE,TITULAR,18,OBLIGATORIO,H_I,AURELIO DIAZ UFANO Y PERAL,2005-07-08,943758196,18.0,Joven (18-29)
4,1,D.N.I.,76180226,TERRONES,VELA,ELKIN JONEL,29/05/2005,HOMBRE,TITULAR,18,OBLIGATORIO,H_I,AURELIO DIAZ UFANO Y PERAL,2005-05-29,915167719,18.0,Joven (18-29)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23152,1,D.N.I.,80505271,VARGAS,TORRES,URSULA ETHEL,21/03/1977,MUJER,TITULAR,47,OBLIGATORIO,H_I,AURELIO DIAZ UFANO Y PERAL,1977-03-21,979950518,47.0,Adulto (30-59)
23153,1,D.N.I.,80491534,QUISPE,HALIRE,WILFREDO,22/06/1979,HOMBRE,TITULAR,44,OBLIGATORIO,H_I,AURELIO DIAZ UFANO Y PERAL,1979-06-22,992990440,44.0,Adulto (30-59)
23154,1,D.N.I.,80274244,CHAVEZ,ORTIZ,MARIA MERCEDES,8/02/1980,MUJER,TITULAR,44,OBLIGATORIO,H_I,AURELIO DIAZ UFANO Y PERAL,1980-08-02,944807272,43.0,Adulto (30-59)
23155,1,D.N.I.,80651704,SERPA,DIAZ,JESUS ALEXANDER DAVID,24/09/1979,HOMBRE,TITULAR,44,OBLIGATORIO,H_I,AURELIO DIAZ UFANO Y PERAL,1979-09-24,923897171,44.0,Adulto (30-59)


In [13]:
# Convertir las series a DataFrames de Pandas
chimbote_30_df = pd.DataFrame(chimbote ['Celular'][chimbote ['Edad'] >= 30], columns=['Celular'])
skrabonja_30_df = pd.DataFrame(skrabonja ['Celular'][skrabonja ['Edad'] >= 30], columns=['Celular'])
torrealva_30_df = pd.DataFrame(torrealva ['Celular'][torrealva ['Edad'] >= 30], columns=['Celular'])
toche_30_df = pd.DataFrame(toche ['Celular'][toche ['Edad'] >= 30], columns=['Celular'])
esperanza_30_df = pd.DataFrame(esperanza ['Celular'][esperanza ['Edad'] >= 30], columns=['Celular'])
reategui_30_df = pd.DataFrame(reategui ['Celular'][reategui ['Edad'] >= 30], columns=['Celular'])

In [14]:
print(chimbote_30_df .shape)
print(skrabonja_30_df .shape)
print(torrealva_30_df .shape)
print(toche_30_df .shape)
print(esperanza_30_df .shape)
print(reategui_30_df .shape)

(646, 1)
(904, 1)
(3423, 1)
(1286, 1)
(3282, 1)
(1344, 1)


In [15]:
chimbote_30_df

,Celular
0,963739966
1,958797886
2,947588640
3,956854376
4,982364871
...,...
644,959638780
645,942000405
646,990378639
647,990683672


In [27]:
# Números adicionales
numeros_adicionales = ['922378050', '940285494', '997686038']

# Convertir las series a DataFrames de Pandas
asegurados1_30_df = pd.DataFrame(asegurados1['Celular'][asegurados1['Edad'] >= 30], columns=['Celular'])
asegurados2_30_df = pd.DataFrame(asegurados2['Celular'][asegurados2['Edad'] >= 30], columns=['Celular'])
asegurados3_30_df = pd.DataFrame(asegurados3['Celular'][asegurados3['Edad'] >= 30], columns=['Celular'])
asegurados4_30_df = pd.DataFrame(asegurados4['Celular'][asegurados4['Edad'] >= 30], columns=['Celular'])
asegurados5_30_df = pd.DataFrame(asegurados5['Celular'][asegurados5['Edad'] >= 30], columns=['Celular'])
asegurados6_30_df = pd.DataFrame(asegurados6['Celular'][asegurados6['Edad'] >= 30], columns=['Celular'])
asegurados7_30_df = pd.DataFrame(asegurados7['Celular'][asegurados7['Edad'] >= 30], columns=['Celular'])

# Agregar números adicionales a cada DataFrame
asegurados1_30_df = asegurados1_30_df.append(pd.DataFrame({'Celular': numeros_adicionales}))
asegurados2_30_df = asegurados2_30_df.append(pd.DataFrame({'Celular': numeros_adicionales}))
asegurados3_30_df = asegurados3_30_df.append(pd.DataFrame({'Celular': numeros_adicionales}))
asegurados4_30_df = asegurados4_30_df.append(pd.DataFrame({'Celular': numeros_adicionales}))
asegurados5_30_df = asegurados5_30_df.append(pd.DataFrame({'Celular': numeros_adicionales}))
asegurados6_30_df = asegurados6_30_df.append(pd.DataFrame({'Celular': numeros_adicionales}))
asegurados7_30_df = asegurados7_30_df.append(pd.DataFrame({'Celular': numeros_adicionales}))

/tmp/ipykernel_3502383/325849119.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  asegurados1_30_df = asegurados1_30_df.append(pd.DataFrame({'Celular': numeros_adicionales}))
/tmp/ipykernel_3502383/325849119.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  asegurados2_30_df = asegurados2_30_df.append(pd.DataFrame({'Celular': numeros_adicionales}))
/tmp/ipykernel_3502383/325849119.py:16: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  asegurados3_30_df = asegurados3_30_df.append(pd.DataFrame({'Celular': numeros_adicionales}))
/tmp/ipykernel_3502383/325849119.py:17: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  asegurados4_30_df = a

In [17]:
chimbote_30_df_m1 = chimbote_30_df
skrabonja_30_df_m1 = skrabonja_30_df
torrealva_30_df_m1 = torrealva_30_df
toche_30_df_m1 = toche_30_df
esperanza_30_df_m1 = esperanza_30_df
reategui_30_df_m1 = reategui_30_df

chimbote_30_df_m2 = chimbote_30_df
skrabonja_30_df_m2 = skrabonja_30_df
torrealva_30_df_m2 = torrealva_30_df
toche_30_df_m2 = toche_30_df
esperanza_30_df_m2 = esperanza_30_df
reategui_30_df_m2 = reategui_30_df

chimbote_30_df_m3 = chimbote_30_df
skrabonja_30_df_m3 = skrabonja_30_df
torrealva_30_df_m3 = torrealva_30_df
toche_30_df_m3 = toche_30_df
esperanza_30_df_m3 = esperanza_30_df
reategui_30_df_m3 = reategui_30_df


In [30]:
asegurados1_30_m1['Mensaje'] = 'ESSALUD. ¡Protege a tu familia del DENGUE! Lava, cepilla o escobilla los recipientes donde sueles almacenar agua: http://www.essalud.gob.pe/whts/'
asegurados2_30_m1['Mensaje'] = 'ESSALUD. ¡Protege a tu familia del DENGUE! Lava, cepilla o escobilla los recipientes donde sueles almacenar agua: http://www.essalud.gob.pe/whts/'
asegurados3_30_m1['Mensaje'] = 'ESSALUD. ¡Protege a tu familia del DENGUE! Lava, cepilla o escobilla los recipientes donde sueles almacenar agua: http://www.essalud.gob.pe/whts/'
asegurados4_30_m1['Mensaje'] = 'ESSALUD. ¡Protege a tu familia del DENGUE! Lava, cepilla o escobilla los recipientes donde sueles almacenar agua: http://www.essalud.gob.pe/whts/'
asegurados5_30_m1['Mensaje'] = 'ESSALUD. ¡Protege a tu familia del DENGUE! Lava, cepilla o escobilla los recipientes donde sueles almacenar agua: http://www.essalud.gob.pe/whts/'
asegurados6_30_m1['Mensaje'] = 'ESSALUD. ¡Protege a tu familia del DENGUE! Lava, cepilla o escobilla los recipientes donde sueles almacenar agua: http://www.essalud.gob.pe/whts/'
asegurados7_30_m1['Mensaje'] = 'ESSALUD. ¡Protege a tu familia del DENGUE! Lava, cepilla o escobilla los recipientes donde sueles almacenar agua: http://www.essalud.gob.pe/whts/'

In [21]:
chimbote_30_df_m2['Mensaje'] = 'ESSALUD. ¿Tienes fiebre, dolor de cabeza o malestar general? ¡Podrías tener DENGUE! Conoce si tienes los síntomas: http://www.essalud.gob.pe/whts/'
skrabonja_30_df_m2['Mensaje'] = 'ESSALUD. ¿Tienes fiebre, dolor de cabeza o malestar general? ¡Podrías tener DENGUE! Conoce si tienes los síntomas: http://www.essalud.gob.pe/whts/'
torrealva_30_df_m2['Mensaje'] = 'ESSALUD. ¿Tienes fiebre, dolor de cabeza o malestar general? ¡Podrías tener DENGUE! Conoce si tienes los síntomas: http://www.essalud.gob.pe/whts/'
toche_30_df_m2['Mensaje'] = 'ESSALUD. ¿Tienes fiebre, dolor de cabeza o malestar general? ¡Podrías tener DENGUE! Conoce si tienes los síntomas: http://www.essalud.gob.pe/whts/'
esperanza_30_df_m2['Mensaje'] = 'ESSALUD. ¿Tienes fiebre, dolor de cabeza o malestar general? ¡Podrías tener DENGUE! Conoce si tienes los síntomas: http://www.essalud.gob.pe/whts/'
reategui_30_df_m2['Mensaje'] = 'ESSALUD. ¿Tienes fiebre, dolor de cabeza o malestar general? ¡Podrías tener DENGUE! Conoce si tienes los síntomas: http://www.essalud.gob.pe/whts/'


In [18]:
chimbote_30_df_m3['Mensaje'] = 'ESSALUD. ¡No te automediques! Si tienes los síntomas del DENGUE, acude al servicio de emergencia del centro de salud más cercano.'
skrabonja_30_df_m3['Mensaje'] = 'ESSALUD. ¡No te automediques! Si tienes los síntomas del DENGUE, acude al servicio de emergencia del centro de salud más cercano.'
torrealva_30_df_m3['Mensaje'] = 'ESSALUD. ¡No te automediques! Si tienes los síntomas del DENGUE, acude al servicio de emergencia del centro de salud más cercano.'
toche_30_df_m3['Mensaje'] = 'ESSALUD. ¡No te automediques! Si tienes los síntomas del DENGUE, acude al servicio de emergencia del centro de salud más cercano.'
esperanza_30_df_m3['Mensaje'] = 'ESSALUD. ¡No te automediques! Si tienes los síntomas del DENGUE, acude al servicio de emergencia del centro de salud más cercano.'
reategui_30_df_m3['Mensaje'] = 'ESSALUD. ¡No te automediques! Si tienes los síntomas del DENGUE, acude al servicio de emergencia del centro de salud más cercano.'

In [40]:
asegurados1_18_m3.tail

<bound method NDFrame.tail of          Celular                                            Mensaje
0      989461960  ESSALUD. ¡No te automediques! Si tienes los sí...
1      922780296  ESSALUD. ¡No te automediques! Si tienes los sí...
2      902592776  ESSALUD. ¡No te automediques! Si tienes los sí...
3      943758196  ESSALUD. ¡No te automediques! Si tienes los sí...
4      915167719  ESSALUD. ¡No te automediques! Si tienes los sí...
...          ...                                                ...
23702  933549016  ESSALUD. ¡No te automediques! Si tienes los sí...
0      922378050  ESSALUD. ¡No te automediques! Si tienes los sí...
1      940285494  ESSALUD. ¡No te automediques! Si tienes los sí...
2      997686038  ESSALUD. ¡No te automediques! Si tienes los sí...
3      971119100  ESSALUD. ¡No te automediques! Si tienes los sí...

[23707 rows x 2 columns]>

In [ ]:
# asegurados1 = pd.read_csv('AURELIO_DIAZ_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
# asegurados2 = pd.read_csv('CHOSICA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
# asegurados3 = pd.read_csv('GUSTAVO_LANATTA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
# asegurados4 = pd.read_csv('HERMANA_MARIA_DONROSE_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
# asegurados5 = pd.read_csv('INDEPENDENCIA_202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
# asegurados6 = pd.read_csv('MARIO_MOLINA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')
# asegurados7 = pd.read_csv('PARAMONGA_P202404.csv', delimiter=';', header=0, encoding='ISO-8859-1', on_bad_lines='warn')

In [22]:
chimbote_30_df_m2.to_excel('CHIMBOTE_21-05.xlsx', index=False)
skrabonja_30_df_m2.to_excel('SKRABONJA_21-05.xlsx', index=False)
torrealva_30_df_m2.to_excel('TORREALVA_20-05.xlsx', index=False)
toche_30_df_m2.to_excel('TOCHE_20-05.xlsx', index=False)
esperanza_30_df_m2.to_excel('ESPERANZA_22-05.xlsx', index=False)
reategui_30_df_m2.to_excel('REATEGUI_22-05.xlsx', index=False)

In [33]:
asegurados1_18_m1.to_excel('AURELIO_DIAZ_09-05.xlsx', index=False)
asegurados2_18_m1.to_excel('CHOSICA_08-05.xlsx', index=False)
asegurados3_18_m1.to_excel('GUSTAVO_LANATTA_07-05.xlsx', index=False)
asegurados4_18_m1.to_excel('HERMANA_MARIA_DONROSE_08-05.xlsx', index=False)
asegurados5_18_m1.to_excel('INDEPENDENCIA_10-05.xlsx', index=False)
asegurados6_18_m1.to_excel('MARIO_MOLINA_07-05.xlsx', index=False)
asegurados7_18_m1.to_excel('PARAMONGA_10-05.xlsx', index=False)

In [30]:
asegurados1_30_m1.to_excel('AURELIO_DIAZ_16-05.xlsx', index=False)
asegurados2_30_m1.to_excel('CHOSICA_15-05.xlsx', index=False)
asegurados3_30_m1.to_excel('GUSTAVO_LANATTA_14-05.xlsx', index=False)
asegurados4_30_m1.to_excel('HERMANA_MARIA_DONROSE_15-05.xlsx', index=False)
asegurados5_30_m1.to_excel('INDEPENDENCIA_17-05.xlsx', index=False)
asegurados6_30_m1.to_excel('MARIO_MOLINA_14-05.xlsx', index=False)
asegurados7_30_m1.to_excel('PARAMONGA_17-05.xlsx', index=False)

In [41]:
asegurados1_18_m3.to_excel('AURELIO_DIAZ_23-05.xlsx', index=False)
asegurados2_18_m3.to_excel('CHOSICA_22-05.xlsx', index=False)
asegurados3_18_m3.to_excel('GUSTAVO_LANATTA_21-05.xlsx', index=False)
asegurados4_18_m3.to_excel('HERMANA_MARIA_DONROSE_22-05.xlsx', index=False)
asegurados5_18_m3.to_excel('INDEPENDENCIA_24-05.xlsx', index=False)
asegurados6_18_m3.to_excel('MARIO_MOLINA_21-05.xlsx', index=False)
asegurados7_18_m3.to_excel('PARAMONGA_24-05.xlsx', index=False)

In [15]:
print(asegurados1_30_df.shape,'AURELIO_DIAZ')
print(asegurados2_30_df.shape,'CHOSICA')
print(asegurados3_30_df.shape,'GUSTAVO_LANATTA')
print(asegurados4_30_df.shape,'HERMANA_MARIA_DONROSE')
print(asegurados5_30_df.shape,'INDEPENDENCIA')
print(asegurados6_30_df.shape,'MARIO_MOLINA')
print(asegurados7_30_df.shape,'PARAMONGA')

(17470, 1) AURELIO_DIAZ
(4292, 1) CHOSICA
(2221, 1) GUSTAVO_LANATTA
(4365, 1) HERMANA_MARIA_DONROSE
(4150, 1) INDEPENDENCIA
(8442, 1) MARIO_MOLINA
(237, 1) PARAMONGA
